In [1]:
import os
import pandas as pd

DATA_DIR = "../data"   # notebook is in notebooks/, data is one level up
for f in sorted(os.listdir(DATA_DIR)):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f"{f:40s} {size_mb:8.1f} MB")

.DS_Store                                     0.0 MB
sample_submission.csv                         0.1 MB
test.csv                                     49.6 MB
train.csv                                   461.0 MB
txs_edgelist.csv                              4.5 MB


In [2]:
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATA_DIR, f), nrows=5)
        n_rows = sum(1 for _ in open(os.path.join(DATA_DIR, f))) - 1
        print(f"\n===== {f}  ({n_rows:,} rows, {df.shape[1]} cols) =====")
        print("First 10 columns:", list(df.columns[:10]))
        print(df.iloc[:3, :8])


===== sample_submission.csv  (15,329 rows, 2 cols) =====
First 10 columns: ['index', 'target']
   index  target
0      0     0.5
1      1     0.5
2      2     0.5

===== test.csv  (15,329 rows, 168 cols) =====
First 10 columns: ['index', 'txId', 'time_step', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7']
   index       txId  time_step    feat_1    feat_2    feat_3    feat_4  \
0      0  200459281         36 -0.172879 -0.058452  0.463609 -0.121970   
1      1  199944769         36 -0.172828  0.339970  1.018602  0.178180   
2      2  198827545         36 -0.172282  0.837998  1.018602  0.553368   

     feat_5  
0 -0.043875  
1 -0.043875  
2 -0.043875  

===== train.csv  (141,772 rows, 168 cols) =====
First 10 columns: ['txId', 'time_step', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8']
        txId  time_step    feat_1    feat_2    feat_3   feat_4    feat_5  \
0  230425980          1 -0.171469 -0.184668 -1.201369 -0.12197 -0.043875

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")

feat_cols = [c for c in train.columns if c.startswith("feat_")]
print("Feature count:", len(feat_cols))
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))

Feature count: 165
Train-only columns: {'label'}
Test-only columns: {'index'}


In [8]:
train["label"] = pd.to_numeric(train["label"], errors="coerce")   # "none" → NaN
print(train["label"].value_counts(dropna=False))
print(f"Illicit rate among labelled: {train['label'].mean():.3%}")

label
NaN    110537
0.0     27591
1.0      3644
Name: count, dtype: int64
Illicit rate among labelled: 11.666%


In [10]:
print("Train time steps:", train.time_step.min(), "→", train.time_step.max())
print("Test time steps: ", test.time_step.min(), "→", test.time_step.max())

# Labels per time step: is the illicit rate stable over time?
print(pd.crosstab(train.time_step, train["label"].fillna("NaN")).to_string())
print(test.time_step.value_counts().sort_index().to_string())

Train time steps: 1 → 35
Test time steps:  36 → 49
label       0.0  1.0   NaN
time_step                 
1          2130   17  5733
2          1099   18  3427
3          1268   11  5342
4          1410   30  4253
5          1874    8  4921
6           480    5  3843
7          1101  102  4845
8          1098   67  3292
9           530  248  4218
10          954   18  5755
11          565  131  3600
12          490   16  1541
13          518  291  3719
14          374   43  1605
15          471  147  3021
16          402  128  2445
17          712   99  2574
18          337   52  1587
19          665   80  2761
20          640  260  3391
21          541  100  2896
22         1605  158  4131
23         1134   53  2978
24          989  137  3466
25          476  118  1720
26          421   96  2006
27          182   24   883
28          199   85  1369
29          845  329  3101
30          441   83  1959
31          604  106  2106
32          981  342  3202
33          418   23  2710
34  

In [6]:
train_ids, test_ids = set(train.txId), set(test.txId)
all_ids = train_ids | test_ids

print("Edges:", len(edges))
print("Edges with both ends in train:", (edges.txId1.isin(train_ids) & edges.txId2.isin(train_ids)).sum())
print("Edges with both ends in test: ", (edges.txId1.isin(test_ids) & edges.txId2.isin(test_ids)).sum())
print("Edges crossing train↔test:   ",
      ((edges.txId1.isin(train_ids) & edges.txId2.isin(test_ids)) |
       (edges.txId1.isin(test_ids) & edges.txId2.isin(train_ids))).sum())
print("Edges touching unknown txIds:", (~edges.txId1.isin(all_ids) | ~edges.txId2.isin(all_ids)).sum())

deg = pd.concat([edges.txId1, edges.txId2]).value_counts()
print("Isolated train nodes:", len(train_ids - set(deg.index)))
print("Isolated test nodes: ", len(test_ids - set(deg.index)))
print(deg.describe())

Edges: 234355
Edges with both ends in train: 163194
Edges with both ends in test:  12724
Edges crossing train↔test:    0
Edges touching unknown txIds: 58437
Isolated train nodes: 0
Isolated test nodes:  0
count    203769.000000
mean          2.300203
std           4.328377
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max         473.000000
Name: count, dtype: float64


In [7]:
print("Missing in train:", train[feat_cols].isna().sum().sum())
print("Missing in test: ", test[feat_cols].isna().sum().sum())

Missing in train: 0
Missing in test:  0


experimenting whether there exists homophily in the training dataset

In [1]:
import pandas as pd

train = pd.read_csv("../data/train.csv")
edges = pd.read_csv("../data/txs_edgelist.csv")
train["label"] = pd.to_numeric(train["label"], errors="coerce")

lab = train.set_index("txId")["label"]
e = edges.assign(l1=edges.txId1.map(lab), l2=edges.txId2.map(lab)).dropna(subset=["l1", "l2"])
print(f"Edges with both ends labelled: {len(e):,}")

base = train["label"].mean()
print(f"Base illicit rate: {base:.1%}")

# If a transaction is illicit, how often is its neighbour illicit (vs. the base rate)?
for direction, (a, b) in {"source → target": ("l1", "l2"), "target ← source": ("l2", "l1")}.items():
    p_ill = e.loc[e[a] == 1, b].mean()
    p_lic = e.loc[e[a] == 0, b].mean()
    print(f"{direction}: neighbour illicit rate | node illicit = {p_ill:.1%}, | node licit = {p_lic:.1%}")

Edges with both ends labelled: 23,900
Base illicit rate: 11.7%
source → target: neighbour illicit rate | node illicit = 56.0%, | node licit = 2.7%
target ← source: neighbour illicit rate | node illicit = 58.9%, | node licit = 3.0%


Testing whether the features are 93 local + rest describing neighbours

In [3]:
import numpy as np
import pandas as pd

train = pd.read_csv("../data/train.csv")
edges = pd.read_csv("../data/txs_edgelist.csv")
feat_cols = [f"feat_{i}" for i in range(1, 166)]

# Undirected neighbour pairs within the training graph (labelled + unlabelled nodes)
pairs = pd.concat([
    edges.rename(columns={"txId1": "node", "txId2": "nbr"}),
    edges.rename(columns={"txId2": "node", "txId1": "nbr"}),
])
X = train.set_index("txId")[feat_cols]
pairs = pairs[pairs.node.isin(X.index) & pairs.nbr.isin(X.index)]

# For every column, compute the max and mean of that column over each node's neighbours
# Reuses X, pairs and feat_cols from the previous cell
nbr_vals = X.loc[pairs.nbr].set_index(pairs.node.values)
g = nbr_vals.groupby(level=0)
nbr_stats = pd.concat({"max": g.max(), "min": g.min(), "mean": g.mean(), "std": g.std()}, axis=1)
nbr_stats = nbr_stats.dropna()          # std is NaN for nodes with a single neighbour
own = X.loc[nbr_stats.index]
src_cols = [c for _, c in nbr_stats.columns]

corr = np.abs(np.corrcoef(own.T.values, nbr_stats.T.values)[:165, 165:])
corr = np.nan_to_num(corr)
for i, c in enumerate(feat_cols):        # ignore "neighbours have similar values of the same column"
    corr[i, [j for j, s in enumerate(src_cols) if s == c]] = 0

best = pd.DataFrame({"best_abs_corr": corr.max(axis=1),
                     "best_match": [f"{a}({b})" for a, b in nbr_stats.columns[corr.argmax(axis=1)]]},
                    index=feat_cols)
best["block"] = [f"feat_{10*(i//10)+1}–{min(10*(i//10)+10, 165)}" for i in range(165)]

print(best.groupby("block", sort=False)["best_abs_corr"]
          .agg(median="median", share_above_0_8=lambda s: (s > 0.8).mean())
          .round(2).to_string())

              median  share_above_0_8
block                                
feat_1–10       0.41              0.4
feat_11–20      0.48              0.4
feat_21–30      0.53              0.0
feat_31–40      0.33              0.1
feat_41–50      0.60              0.5
feat_51–60      0.52              0.0
feat_61–70      0.32              0.0
feat_71–80      0.71              0.3
feat_81–90      0.48              0.3
feat_91–100     0.47              0.4
feat_101–110    0.28              0.2
feat_111–120    0.20              0.3
feat_121–130    0.22              0.2
feat_131–140    0.56              0.2
feat_141–150    0.34              0.0
feat_151–160    0.27              0.0
feat_161–165    0.13              0.0
